<a href="https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/VIX_FINAL_FEATURES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VIX Final Features v1 — dataset lourd partagé (1/4)

**Rôle dans la campagne finale.** Ce notebook est le **premier** d'une série de 4, conçus pour être lancés **séparément** (onglets Colab différents) plutôt que comme un seul run monolithique :
1. **`VIX_FINAL_FEATURES`** (ici) — charge l'univers lourd (421 tickers) + FRED, construit toutes les features causales (Kalman, HMM, EGARCH/GJR, Heston, VRP, Hawkes, spike/structure par terme), découvre les **interactions inter-tickers** (2 passes SHAP, méthodologie du rapport §3.9), et **pousse le dataset enrichi** sur GitHub.
2. **`VIX_FINAL_ML_SCAN`** — recharge ce dataset, scan complet (6 horizons × 4 régimes × N=5-15 × 5 samplers × 5 algos dont CatBoost) walk-forward, checkpointé/résumable.
3. **`VIX_FINAL_TFT`** — recharge ce dataset, TFT sur tous les horizons × régimes, checkpointé/résumable.
4. **`VIX_FINAL_OPTUNA`** — recharge les résultats du scan ML, affine les meilleures configs par hyperparameter tuning (Optuna, TPE, MedianPruner).

**Pourquoi séparer** : le scan ML complet représente ~66 000 entraînements (plusieurs dizaines d'heures) — largement au-delà d'une session Colab. Recalculer le feature engineering lourd (téléchargement + Kalman/HMM/EGARCH/Heston à chaque run) dans chaque notebook serait un gâchis de temps ; il n'est fait **qu'ici**, une fois, et partagé via GitHub.

**Ce qui n'est PAS repris** : le filtre particulaire (0 sélection dans les scans précédents, cf. `VIX_SPIKE_SCAN v2`) — abandonné, sans perte de généralité démontrée.

**Prérequis pour le partage automatique** : secret Colab `GITHUB_TOKEN` (voir README du repo). Sans lui, le dataset reste local à ce runtime — il faudra le transférer manuellement aux 3 autres notebooks.


In [ ]:
import subprocess, sys
pkgs = ['xgboost','yfinance','pandas_datareader','arch',
        'pykalman','hmmlearn','shap','statsmodels','pyarrow']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


In [ ]:
import os, time, json, warnings, random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import shap
from scipy.stats import multivariate_normal
from scipy.special import logsumexp
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier
import statsmodels.api as sm
from arch import arch_model
from pykalman import KalmanFilter
from hmmlearn import hmm as hmmlib

SEED = 42; random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_NAME = 'VIX_FINAL_FEATURES'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'start_date': '2000-01-01',
    'flat_thr': 0.003,
    'n_wf_folds': 5,
    'min_train_frac': 0.40,
    'yf_coverage': 0.85,
    'spike_coverage': 0.30,
    'shap_sample': 500,
    'pool_prefilter': 450,
    # Interactions inter-tickers (report §3.9) : passe 1 top-40, passe 2 sur top-20 (6 types), re-sélection top-30
    'interact_top_base': 40,
    'interact_top_pairs': 20,
    'interact_final_n': 30,
    # Cible pilote pour la découverte des interactions (une seule décision structurelle, pas par horizon)
    'pilot_horizon': 5,
}
TARGET_COL = 'VIX_Amplitude_Class'

# Univers lourd (421 tickers, repris de VIX_SPIKE_SCAN v2)
YF_TICKERS = """
^GSPC ^IXIC ^DJI ^RUT ^VIX ^VXN ^OVX ^GVZ ^EVZ ^VVIX ^FTSE ^N225
^HSI ^GDAXI ^FCHI ^STOXX50E SPY QQQ TLT GLD USO UUP FXE FXY
HYG LQD AAPL MSFT GOOG AMZN NVDA TSLA JPM JNJ V MA
PG UNH HD KO PEP T SMFG DIS XOM CVX BAC WMT
VZ CSCO ORCL CRM AMD NFLX ADBE INTC CMCSA PFE ABT LLY
DHR COST CMG SBUX MCD ACN PYPL QCOM TXN BA GE BABA
^BVSP ^AXJO ^AORD ^IBEX VXX UVXY VIXY SVXY VXZ VIXM XLK XLF
XLE XLV XLU XLP XLI XLY XLRE XLB XLC GOOGL META AVGO
ASML WFC GS BLK SCHW MS COF BAX AXP EQR PLD AMT
EQIX CCI PSA ABBV MRK BMY AMGN GILD BNTX MRNA CRSP VRTX
ILMN DXCM TDOC CI HUM RTX LMT NOC GD CAT DE ITT
PAYX CTAS MMM HON ETN EMR OTIS JCI PTC SMCI COP SLB
EOG MPC PSX VLO PM MO BTI BP TTE ENB MET ADM
MKC SJM CPB GIS MDLZ NSRGY TAP BDX CLX CL UL YUM
QSR DPZ BLMN NWL RRR DASH LYFT UBER TGT M LOW ROST
BBY NEE DUK SO AEP EXC SRE ES XEL PPL TMUS CHTR
VOD TM LOGI NET DDOG SLV UNG DBC DBA GDX GDXJ PDBC
CORN SOYB IEF SHY SHV BIL AGG BND JNK VCIT VCSH EMB
MBB TIP BNDX HYLD PFFA EWJ EWG EWU EWA EWH EWL EWP
EWI EWQ EWT EWY EWZ EWC EWS EWM FXI MCHI IEMG EEM
VEA INDA EPI ASHR TUR EIDO EPOL EZA GXG EGRX FXB FXA
FXC FXF CEW CYB BZF FXD FXN GBTC COIN MSTR BITO MARA
RIOT CLSK CIFR CORZ IWM IVV VTI VOO VV VTV VUG VB
SCHD VIG HDV NOBL DGRO QUAL VLUE VYMI JEPI XYLD QYLD RYLD
ARKK XBI SOXX IBB IYT XHB KRE KBE ITA XOP OIH IYM
PCAR DAL AAL UAL LUV ICLN TAN MTUM USMV SPLV RSP EUSA
EEMV VNQ IYR REM SPG AVB COLD DLR REXR HII LDOS EBAY
MELI SHOP SE PDD JD VIPS UPST RBLX SNOW CRWD ZM ROKU
PINS SNAP SPCE
IWD IWF IWN IWO IWS IWP IWR SIZE QUS FNDX FNDA SPHB
XME XRT XLG PPA IHI IHF IGV FDN SKYY HACK ARKG ARKW ARKF
GUNR MOO WOOD LIT REMX URA COPX SIL PPLT PALL JJC CPER
BKLN SRLN FLOT SJNK ANGL HYEM EMHY PCY BWX WIP VWOB
KWEB CQQQ EWW EWD EWN EWO ENZL THD VNM FLKR GREK ARGT
DVY SDY VYM SPYD FVD DON DES PEY RDVY
XLC IYW VGT SMH PSI XSD IGM
KIE IAI XBI PBW QCLN FAN
""".split()

SPIKE_TICKERS = "^SKEW ^VIX3M ^VIX9D ^VIX1D ^VIX6M ^VXV ^PCALL".split()
FRED_SERIES = {'NFCI': 'NFCI', 'STLFSI': 'STLFSI4', 'T10Y2Y': 'T10Y2Y', 'EFFR': 'EFFR'}

print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | univers={len(YF_TICKERS)} tickers + spike | "
      f"interactions: top{CONFIG['interact_top_base']}->pairs top{CONFIG['interact_top_pairs']}->final top{CONFIG['interact_final_n']}")


In [ ]:
def load_data(start=CONFIG['start_date']):
    t0 = time.time()
    # --- univers lourd (filtre de couverture standard) ---
    raw = yf.download(YF_TICKERS, start=start, auto_adjust=True, progress=False)['Close']
    raw.columns = [c.replace('^', 'IDX_').replace('-', '_') for c in raw.columns]
    raw = raw.loc[:, raw.notna().mean() >= CONFIG['yf_coverage']].ffill().dropna(how='all')
    print(f"  YF lourd: {raw.shape[0]}j × {raw.shape[1]} retenus (couv≥{CONFIG['yf_coverage']}) ({time.time()-t0:.1f}s)")

    # --- tickers spike/tail-risk : téléchargement ROBUSTE ticker par ticker ---
    # [FIX v2] En v1 le batch yf.download échouait en bloc si un seul ticker
    # posait problème (^SKEW, ^VIX9D perdus). Ici chaque ticker est téléchargé
    # isolément ; un échec ne fait perdre que ce ticker. ^VXV sert de secours
    # longue-histoire pour la structure par terme 3 mois si ^VIX3M manque.
    spike_added = []
    for tk in SPIKE_TICKERS:
        col = tk.replace('^', 'IDX_').replace('-', '_')
        if col in raw.columns:
            continue
        try:
            s = yf.download(tk, start=start, auto_adjust=True, progress=False)['Close']
            if isinstance(s, pd.DataFrame):
                s = s.iloc[:, 0]
            s = s.reindex(raw.index).ffill()
            if s.notna().mean() >= CONFIG['spike_coverage']:
                raw[col] = s; spike_added.append(col)
            else:
                print(f"  YF spike: {tk} ignoré (couverture {s.notna().mean():.0%} < {CONFIG['spike_coverage']:.0%})")
        except Exception as e:
            print(f"  [WARN] spike {tk}: {str(e)[:80]}")
    # secours : si pas de VIX3M mais ^VXV présent, l'utiliser comme VIX3M
    if not any('VIX3M' in c for c in raw.columns) and 'IDX_VXV' in raw.columns:
        raw['IDX_VIX3M'] = raw['IDX_VXV']; spike_added.append('IDX_VIX3M(via VXV)')
    print(f"  YF spike ajoutés: {spike_added if spike_added else 'aucun'}")

    # --- FRED ---
    fred_list = []
    for name, sid in FRED_SERIES.items():
        try:
            s = web.DataReader(sid, 'fred', start).squeeze(); s.name = f'FRED_{name}'
            fred_list.append(s)
        except Exception as e: print(f"  [WARN] {sid}: {e}")
    if fred_list:
        raw = pd.concat([raw, pd.concat(fred_list, axis=1).reindex(raw.index, method='ffill')], axis=1)
    print(f"  Total: {raw.shape} ({time.time()-t0:.1f}s)")
    return raw

df_raw = load_data()
all_dates = df_raw.dropna(how='all').index.sort_values()
VIX_COL = [c for c in df_raw.columns if ('IDX_VIX' in c or c.endswith('_VIX')) and 'VXN' not in c
           and 'VVIX' not in c and 'VIX3M' not in c and 'VIX9D' not in c][0]
SPX_COLS = [c for c in df_raw.columns if 'GSPC' in c or c == 'SPY']
SPX_COL = SPX_COLS[0] if SPX_COLS else None

# Bornes des folds walk-forward (expanding window)
n_obs = len(all_dates)
first_cut = int(n_obs * CONFIG['min_train_frac'])
test_span = (n_obs - first_cut) // CONFIG['n_wf_folds']
FOLD_CUTS = [first_cut + k * test_span for k in range(CONFIG['n_wf_folds'] + 1)]
FOLD_CUTS[-1] = n_obs
FIT_IDX = FOLD_CUTS[0]

print(f"VIX={VIX_COL} | SPX={SPX_COL}")
for k in range(CONFIG['n_wf_folds']):
    print(f"  Fold {k+1}: train → {all_dates[FOLD_CUTS[k]-1].date()} | "
          f"test {all_dates[FOLD_CUTS[k]].date()} → {all_dates[FOLD_CUTS[k+1]-1].date()}")


## Fondements théoriques des features (Kalman, HMM, EGARCH/GJR, Heston, VRP, Hawkes)

La cellule suivante (`build_features`) construit plusieurs familles de features issues de modèles de séries temporelles financières. Pour chacune : le principe, l'objectif dans ce pipeline, et les fondements mathématiques.

### Filtre de Kalman
**Principe** — Estimateur récursif à variance minimale de l'état caché d'un système linéaire-gaussien à partir d'observations bruitées : état $x_t = F x_{t-1} + w_t$, observation $y_t = H x_t + v_t$, avec $w_t \sim \mathcal N(0,Q)$, $v_t \sim \mathcal N(0,R)$. Deux étapes à chaque pas : **prédiction** ($\hat x_{t|t-1} = F\hat x_{t-1}$) puis **mise à jour** par le gain de Kalman $K_t = P_{t|t-1}H^\top(HP_{t|t-1}H^\top+R)^{-1}$, qui pondère l'innovation $y_t - H\hat x_{t|t-1}$.
**Objectif ici** — Le VIX est modélisé comme une marche aléatoire bruitée ($F=H=1$) : le filtre en extrait un niveau "sous-jacent" débruité de façon **causale** (chaque estimation ne dépend que du passé). Le résidu (VIX observé − VIX filtré) sert de feature d'écart au régime.
**Point de vigilance** — Le lisseur RTS (`kf.smooth()`, forward-backward) utilise aussi les observations *futures* : c'est non causal, donc écarté en walk-forward (cf. fix v2).

### HMM (Hidden Markov Model) — détection de régime
**Principe** — Chaîne de Markov cachée à états discrets (ici 2 : "calme"/"stress") qui gouverne la loi des observations à chaque instant. Paramètres : probabilités initiales $\pi$, matrice de transition $A$, lois d'émission par état (ici gaussiennes). Apprentissage par EM (Baum-Welch) : l'étape E calcule les probabilités a posteriori des états via l'algorithme forward-backward, l'étape M réestime $\pi, A$ et les émissions.
**Objectif ici** — Estimer $P(\text{état stress}_t \mid \text{observations})$ comme feature de régime de volatilité.
**Fondements / cours de référence** — [Probabilistic Graphical Models (Master MVA)](https://www.master-mva.com/cours/probabilistic-graphical-models/) : distinction clé entre **filtrage** (forward seul, $P(z_t\mid x_{1:t})$, causal) et **lissage** (forward-backward, $P(z_t\mid x_{1:T})$, non causal — utilise le futur). `hmmlearn.predict_proba()` fait du lissage : c'est la fuite corrigée en v2 via `hmm_filtered_proba()` (forward uniquement).

### EGARCH / GJR-GARCH — volatilité conditionnelle
**Principe** — Modélisent la variance conditionnelle $\sigma_t^2$ d'une série de rendements pour capter le *clustering* de volatilité et l'**effet de levier** (une baisse augmente plus la volatilité future qu'une hausse de même ampleur). GARCH(1,1) : $\sigma_t^2=\omega+\alpha r_{t-1}^2+\beta\sigma_{t-1}^2$. **EGARCH** (Nelson, 1991) modélise $\log\sigma_t^2$ (positivité garantie sans contrainte sur les paramètres) avec un terme asymétrique en $r_{t-1}/\sigma_{t-1}$. **GJR-GARCH** (Glosten–Jagannathan–Runkle, 1993) ajoute un terme indicatrice sur les rendements négatifs.
**Objectif ici** — Paramètres estimés sur les rendements du S&P 500 en train uniquement, puis le chemin de variance conditionnelle est recalculé de façon causale (récursion sur les rendements passés) sur tout l'historique via `arch_model.fix()` — fixe le bug v1 où les features de test restaient plates.

### Modèle de Heston (proxy)
**Principe** — Modèle à volatilité stochastique : la variance instantanée suit un processus CIR (Cox–Ingersoll–Ross) à retour à la moyenne $dv_t=\kappa(\theta-v_t)dt+\xi\sqrt{v_t}\,dW_t$, corrélé au prix ($\rho$). $\kappa$ = vitesse de retour à la moyenne, $\theta$ = variance long-terme, $\xi$ = vol-of-vol.
**Objectif ici** — Sans données d'options pour calibrer Heston par ajustement de surface de volatilité implicite, on construit des **proxies** : $\theta$ via variance réalisée glissante, $\xi$ via le VVIX (vol du VIX), $\kappa$ via la demi-vie d'un AR(1) glissant sur le VIX (relation $HL=\ln 2/\kappa$). L'espérance conditionnelle de variance à horizon $h$, $\mathbb E[v_{t+h}]=\theta+(v_t-\theta)e^{-\kappa h}$, sert de feature de "retour à la moyenne attendu".

### VRP (Variance Risk Premium)
**Principe** — Écart entre variance implicite (ici $(\text{VIX}/100)^2$) et variance réalisée future attendue : $VRP_t = IV_t^2-\mathbb E[RV_{t\to t+h}]$ — mesure la prime que le marché paie pour se couvrir contre le risque de variance.
**Objectif ici** — $\mathbb E[RV]$ est estimée par régression OLS de la variance réalisée future sur des variances réalisées passées à plusieurs échelles (1j/5j/22j), une version simplifiée de l'approche **HAR** (Heterogeneous AutoRegressive, Corsi 2009).

### Processus de Hawkes (proxy d'intensité de sauts)
**Principe** — Processus ponctuel auto-excitant : chaque événement (ici un "saut" de rendement du VIX) augmente temporairement l'intensité d'occurrence de futurs événements, avec décroissance exponentielle $\lambda(t)=\mu+\sum_{t_i<t}\alpha e^{-\beta(t-t_i)}$ — modélise le *clustering* temporel des chocs de volatilité.
**Objectif ici** — Proxy simplifié (pas de MLE complet du processus ponctuel) : intensité pondérée par décroissance exponentielle du temps écoulé depuis les sauts passés.


In [ ]:
def hmm_filtered_proba(model, X):
    """Probabilités d'état HMM causales (forward filtering P(state_t | obs_1..t)),
    contrairement à predict_proba() qui fait du forward-backward (smoothing) et
    utilise donc des observations futures -> fuite de données en walk-forward."""
    n = len(X)
    n_states = model.n_components
    logframe = np.zeros((n, n_states))
    for s in range(n_states):
        cov = model.covars_[s]
        if cov.ndim == 2:
            cov = cov + np.eye(cov.shape[0]) * 1e-6
        logframe[:, s] = multivariate_normal.logpdf(X, mean=model.means_[s], cov=cov)
    log_start = np.log(model.startprob_ + 1e-300)
    log_trans = np.log(model.transmat_ + 1e-300)
    fwd = np.zeros((n, n_states))
    fwd[0] = log_start + logframe[0]
    fwd[0] -= logsumexp(fwd[0])
    for t in range(1, n):
        for j in range(n_states):
            fwd[t, j] = logsumexp(fwd[t - 1] + log_trans[:, j]) + logframe[t, j]
        fwd[t] -= logsumexp(fwd[t])
    return np.exp(fwd)


def build_features(df_raw, vix_col, spx_col, split_idx):
    t0 = time.time(); feats = {}
    vix = df_raw[vix_col].replace([np.inf, -np.inf], np.nan).ffill().bfill()
    vix_ret = np.log(vix / vix.shift(1)).replace([np.inf, -np.inf], np.nan).fillna(0)
    spx_ret = pd.Series(0., index=df_raw.index)
    if spx_col:
        spx = df_raw[spx_col].ffill().bfill()
        spx_ret = np.log(spx / spx.shift(1)).fillna(0)

    # Rendements multi-horizons pour tous les tickers
    print(f"  [FEAT] Rendements ({df_raw.shape[1]} séries)...")
    for col in df_raw.columns:
        s = df_raw[col].replace([np.inf, -np.inf], np.nan).ffill().bfill()
        lr = np.log(s / s.shift(1)).replace([np.inf, -np.inf], np.nan)
        for w in [1, 5, 20]:
            feats[f'{col}_ret_{w}d'] = np.log(s / s.shift(w)).replace([np.inf, -np.inf], np.nan)
        feats[f'{col}_vol_20d'] = lr.rolling(20, min_periods=10).std()
        mu = s.rolling(60, min_periods=30).mean(); sd = s.rolling(60, min_periods=30).std().replace(0, np.nan)
        feats[f'{col}_zscore_60d'] = (s - mu) / sd

    # VIX features
    for w in [5, 10, 20]:
        ma = vix.rolling(w, min_periods=w // 2).mean(); sd = vix.rolling(w, min_periods=w // 2).std().replace(0, np.nan)
        feats[f'vix_zscore_{w}d'] = (vix - ma) / sd
        feats[f'vix_vs_ma{w}'] = (vix - ma) / ma.replace(0, np.nan)
    feats['vix_level'] = vix; feats['vix_ma_20'] = vix.rolling(20, min_periods=10).mean()
    for w in [5, 10]: feats[f'vix_vol_of_vol_{w}d'] = vix_ret.rolling(w, min_periods=w // 2).std()
    for w in [2, 3, 5]: feats[f'vix_momentum_{w}d'] = vix.pct_change(w)
    feats['vix_acceleration_1d'] = vix_ret - vix_ret.shift(1)
    feats['vix_acceleration_3d'] = vix_ret - vix_ret.shift(3)
    feats['vix_erratic_ratio'] = vix_ret.abs().rolling(5, min_periods=3).max() / vix_ret.abs().rolling(5, min_periods=3).mean().replace(0, np.nan)
    feats['vix_vol_ratio_5_60'] = vix_ret.rolling(5, min_periods=3).std() / vix_ret.rolling(60, min_periods=30).std().replace(0, np.nan)
    feats['vix_max_abs_ret_5d'] = vix_ret.abs().rolling(5, min_periods=3).max()
    feats['vix_mean_abs_ret_5d'] = vix_ret.abs().rolling(5, min_periods=3).mean()
    if spx_col:
        spx = df_raw[spx_col].ffill().bfill()
        feats['spx_drawdown_252d'] = (spx - spx.rolling(252, min_periods=126).max()) / spx.rolling(252, min_periods=126).max().replace(0, np.nan)
        feats['spx_vol_5d'] = spx_ret.rolling(5, min_periods=3).std()
        feats['spx_abs_ret_max_5d'] = spx_ret.abs().rolling(5, min_periods=3).max()
        feats['spx_momentum_3d'] = spx.pct_change(3)
        feats['vix_spx_corr_30d'] = vix_ret.rolling(30, min_periods=15).corr(spx_ret)

    # EGARCH + GJR-GARCH
    # [FIX v2] Params estimés UNIQUEMENT sur le train (pas de fuite), puis appliqués
    # via .fix() sur toute la série pour obtenir un chemin de variance conditionnelle
    # causal (récursif, ne dépend que du passé) sur train+test. v1 utilisait
    # res.forecast(start=0) qui ne s'étend pas au-delà de l'échantillon d'estimation
    # -> les features EGARCH du test étaient plates (ffill de la dernière valeur train).
    print(f"  [FEAT] EGARCH ({time.time() - t0:.0f}s)...")
    try:
        sp_tr = (spx_ret.iloc[:split_idx] * 100)
        sp_full = (spx_ret * 100).replace([np.inf, -np.inf], np.nan).fillna(0)
        am = arch_model(sp_tr, vol='EGARCH', p=1, q=1, dist='skewt', rescale=False)
        res = am.fit(disp='off', show_warning=False)
        am_full = arch_model(sp_full, vol='EGARCH', p=1, q=1, dist='skewt', rescale=False)
        res_full = am_full.fix(res.params)
        cv = (res_full.conditional_volatility.pow(2) / 10000).reindex(df_raw.index).replace([np.inf, -np.inf], np.nan)
        feats['egarch_condvar'] = cv; feats['egarch_delta'] = cv.diff()
        for h in [1, 3, 5]:
            feats[f'EGARCH_SPX_condvar_h{h}'] = cv; feats[f'EGARCH_SPX_delta_h{h}'] = cv.diff()
        # GJR-GARCH
        am2 = arch_model(sp_tr, vol='GARCH', p=1, o=1, q=1, dist='skewt', rescale=False)
        res2 = am2.fit(disp='off', show_warning=False)
        am2_full = arch_model(sp_full, vol='GARCH', p=1, o=1, q=1, dist='skewt', rescale=False)
        res2_full = am2_full.fix(res2.params)
        cv2 = (res2_full.conditional_volatility.pow(2) / 10000).reindex(df_raw.index).replace([np.inf, -np.inf], np.nan)
        feats['gjr_condvar'] = cv2; feats['gjr_delta'] = cv2.diff()
        print(f"    EGARCH+GJR OK (paramètres train-only, chemin causal complet)")
    except Exception as e: print(f"    [WARN] GARCH: {e}")

    # Kalman (causal ; +1j shift anti-leakage)
    # [FIX v2] v1 calculait aussi kf.smooth() (RTS smoother, non causal : l'estimé à la
    # date t dépend d'observations futures jusqu'à la fin de l'échantillon) et l'utilisait
    # pour VIX_Innovation -> fuite. On ne garde que le filtre forward (causal).
    print(f"  [FEAT] Kalman ({time.time() - t0:.0f}s)...")
    try:
        vc = vix.interpolate('linear').ffill().bfill().astype(float)
        kf = KalmanFilter(transition_matrices=[[1.]], observation_matrices=[[1.]],
                        initial_state_mean=[float(vc.iloc[0])],
                        initial_state_covariance=[[1.]],
                        em_vars=['transition_covariance', 'observation_covariance'])
        kf = kf.em(vc.iloc[:split_idx].values.reshape(-1, 1), n_iter=20)
        smf, _ = kf.filter(vc.values.reshape(-1, 1))
        kf_f = pd.Series(smf[:, 0], index=df_raw.index)
        feats['VIX_Residual'] = (vc - kf_f).shift(1).replace([np.inf, -np.inf], np.nan)
        feats['VIX_Innovation'] = (vc - kf_f.shift(1)).replace([np.inf, -np.inf], np.nan)
        feats['kalman_residual'] = feats['VIX_Residual']
        feats['kalman_innovation'] = feats['VIX_Innovation']
        feats['kalman_filtered'] = kf_f
        print(f"    Kalman OK (Q={kf.transition_covariance[0,0]:.5f}, filtre causal uniquement)")
    except Exception as e: print(f"    [WARN] Kalman: {e}")

    # HMM K=2 (probabilités filtrées causales, pas smoothées)
    # [FIX v2] mh.predict_proba()/mh.predict() en v1 utilisaient l'algorithme forward-
    # backward sur train+test entier -> l'état "stress" à la date t incorporait de
    # l'information sur les rendements futurs (t+1..T). On calcule maintenant les
    # probabilités filtrées (forward only) via hmm_filtered_proba(), strictement causales.
    print(f"  [FEAT] HMM ({time.time() - t0:.0f}s)...")
    try:
        rv5 = vix_ret.pow(2).rolling(5, min_periods=3).mean()
        mu_t = vix.iloc[:split_idx].mean(); sd_t = vix.iloc[:split_idx].std()
        vix_n = (vix - mu_t) / (sd_t if sd_t > 1e-8 else 1)
        X_hmm = pd.DataFrame({'r': vix_ret, 'v': np.sqrt(rv5.clip(0)), 'l': vix_n}).dropna()
        mh = hmmlib.GaussianHMM(n_components=2, covariance_type='full', n_iter=200, random_state=SEED)
        mh.fit(X_hmm.iloc[:split_idx].values)
        st = mh.predict(X_hmm.iloc[:split_idx].values)
        rv_v = [rv5.reindex(X_hmm.index[:split_idx]).values[st == s].mean() if (st == s).any() else 0 for s in range(2)]
        ss = int(np.argmax(rv_v))
        pr = hmm_filtered_proba(mh, X_hmm.values)
        p_stress = pd.Series(pr[:, ss], index=X_hmm.index).reindex(df_raw.index)
        feats['P_stress_HMM'] = p_stress; feats['hmm_p_stress'] = p_stress
        feats['hmm_state'] = pd.Series(pr.argmax(axis=1), index=X_hmm.index).reindex(df_raw.index)
        print(f"    HMM OK (état stress={ss}, probabilités filtrées causales)")
    except Exception as e: print(f"    [WARN] HMM: {e}")

    # Heston proxies + espérances conditionnelles
    print(f"  [FEAT] Heston ({time.time() - t0:.0f}s)...")
    try:
        v0 = (vix / 100).pow(2); theta = vix_ret.pow(2).rolling(60, min_periods=30).mean()
        vvix_c = [c for c in df_raw.columns if 'VVIX' in c]
        xi = (df_raw[vvix_c[0]].ffill().bfill() / 100) if vvix_c else vix_ret.rolling(20).std()
        xi = xi.reindex(df_raw.index, method='ffill').replace([np.inf, -np.inf], np.nan)
        rho = vix_ret.rolling(30, min_periods=15).corr(spx_ret)
        rho_60 = vix_ret.rolling(60, min_periods=30).corr(spx_ret)
        # Kappa via half-life AR(1) rolling 252j
        def rolling_kappa(s, w=252):
            k = pd.Series(np.nan, index=s.index); sf = s.ffill().bfill()
            for i in range(w, len(sf)):
                try:
                    b = np.corrcoef(sf.iloc[i - w:i].values, sf.iloc[i - w + 1:i + 1].values)[0, 1]
                    if np.isfinite(b) and 0 < abs(b) < 0.9999:
                        hl = -np.log(2) / np.log(abs(b))
                        if np.isfinite(hl) and hl > 0: k.iloc[i] = np.log(2) / hl
                except: pass
            return k.replace([np.inf, -np.inf], np.nan)
        kappa = rolling_kappa(vix)
        feats['heston_v0'] = v0; feats['heston_theta'] = theta
        feats['heston_xi'] = xi; feats['heston_rho'] = rho; feats['heston_rho_60'] = rho_60
        feats['heston_kappa'] = kappa
        feats['heston_feller'] = (2 * kappa * theta) / xi.pow(2).replace(0, np.nan)
        feats['heston_v0_minus_theta'] = v0 - theta
        mu_xi = xi.iloc[:split_idx].mean(); sd_xi = xi.iloc[:split_idx].std()
        feats['heston_xi_zscore'] = (xi - mu_xi) / (sd_xi if sd_xi > 1e-8 else 1)
        feats['heston_xi_ma5'] = xi.rolling(5, min_periods=3).mean()
        mu_k = kappa.iloc[:split_idx].mean(); sd_k = kappa.iloc[:split_idx].std()
        feats['heston_kappa_zscore'] = (kappa - mu_k) / (sd_k if sd_k > 1e-8 else 1)
        for h in [1, 3, 5, 7, 10]:
            ev = (theta + (v0 - theta) * np.exp(-kappa * h)).replace([np.inf, -np.inf], np.nan)
            var_ev = (v0 * xi ** 2 * np.exp(-kappa * h) * (1 - np.exp(-kappa * h)) / kappa.replace(0, np.nan)
                    + theta * xi ** 2 * (1 - np.exp(-kappa * h)) ** 2 / (2 * kappa.replace(0, np.nan))).replace([np.inf, -np.inf], np.nan)
            feats[f'heston_ev_h{h}'] = ev
            feats[f'heston_spread_h{h}'] = v0 - ev
            feats[f'heston_vol_h{h}'] = np.sqrt(ev.clip(lower=0)) * 100
            feats[f'heston_var_ev_h{h}'] = var_ev
        print(f"    Heston OK (xi_mean={xi.iloc[:split_idx].mean():.4f})")
    except Exception as e: print(f"    [WARN] Heston: {e}")

    # VRP
    # [FIX v2] La cible d'entraînement de la régression (rv_tgt) regarde 22j dans le
    # futur. En v1, htr=hdf.iloc[:split_idx] incluait des lignes dont la cible dépassait
    # split_idx et lisait donc des données de test -> fuite dans les coefficients OLS.
    # On tronque désormais l'échantillon d'entraînement de 22j avant le split.
    print(f"  [FEAT] VRP ({time.time() - t0:.0f}s)...")
    try:
        rv1 = vix_ret.pow(2).replace([np.inf, -np.inf], np.nan)
        rv5d = rv1.rolling(5, min_periods=3).mean(); rv22d = rv1.rolling(22, min_periods=10).mean()
        rv_tgt = rv1.shift(-22).rolling(22, min_periods=11).mean()
        hdf = pd.DataFrame({'rv1': rv1, 'rv5': rv5d, 'rv22': rv22d, 'y': rv_tgt}).dropna().replace([np.inf, -np.inf], np.nan).dropna()
        train_cutoff = max(split_idx - 22, 1)
        htr = hdf.iloc[:train_cutoff]
        Xh = sm.add_constant(htr[['rv1', 'rv5', 'rv22']], has_constant='add')
        hm = sm.OLS(htr['y'], Xh).fit()
        Xf = sm.add_constant(hdf[['rv1', 'rv5', 'rv22']], has_constant='add').fillna(0)
        rv_pred = hm.predict(Xf).reindex(df_raw.index).fillna(0)
        vrp = (vix / 100).pow(2) - rv_pred
        mu_v = vrp.iloc[:split_idx].mean(); sd_v = vrp.iloc[:split_idx].std()
        feats['VRP'] = vrp; feats['VRP_zscore'] = (vrp - mu_v) / (sd_v if sd_v > 1e-8 else 1)
        feats['VRP_ma5'] = vrp.rolling(5, min_periods=3).mean()
        print(f"    VRP OK (R²={hm.rsquared:.4f}, cible tronquée avant le split)")
    except Exception as e: print(f"    [WARN] VRP: {e}")

    # Jump + Hawkes
    print(f"  [FEAT] Jump+Hawkes ({time.time() - t0:.0f}s)...")
    sig60 = vix_ret.rolling(60, min_periods=30).std()
    is_j = (vix_ret.abs() > 3 * sig60).astype(float)
    feats['jump_intensity_20d'] = is_j.rolling(20, min_periods=10).mean()
    feats['jump_intensity_60d'] = is_j.rolling(60, min_periods=30).mean()
    try:
        sig_hw = vix_ret.rolling(30, min_periods=15).std()
        jt = vix_ret.index[vix_ret.abs() > 2 * sig_hw]
        hw = pd.Series(0., index=vix_ret.index)
        for i, t in enumerate(vix_ret.index):
            past = jt[jt < t]
            hw.iloc[i] = 0.3 + 0.3 * float(np.sum(np.exp(-0.1 * np.array([(t - tj).days for tj in past], dtype=float)))) if len(past) else 0.3
        mu_hw = hw.iloc[:split_idx].mean(); sd_hw = hw.iloc[:split_idx].std()
        feats['hawkes_intensity'] = hw; feats['hawkes_zscore'] = (hw - mu_hw) / (sd_hw if sd_hw > 1e-8 else 1)
    except Exception as e: print(f"    [WARN] Hawkes: {e}")

    # Implied correlation proxy
    try:
        sec = [c for c in df_raw.columns if any(s in c for s in ['XLK', 'XLF', 'XLE', 'XLV', 'XLU', 'XLB', 'XLI', 'XLY'])]
        if sec:
            vix_sq = (vix / 100).pow(2); w = 1. / len(sec)
            sv_sum = sum(w ** 2 * np.log(df_raw[c].ffill() / df_raw[c].ffill().shift(1)).rolling(21, min_periods=10).std().pow(2) for c in sec)
            feats['impl_corr_proxy'] = (vix_sq - sv_sum).clip(-1, 1)
    except: pass

    df_feat = pd.DataFrame(feats, index=df_raw.index).replace([np.inf, -np.inf], np.nan)
    df_full = pd.concat([df_raw, df_feat], axis=1)
    df_full = df_full.loc[:, ~df_full.columns.duplicated()]
    print(f"  [FEAT] Total: {df_full.shape[1]} cols ({time.time() - t0:.0f}s)")
    return df_full


# [WF] Les estimateurs internes (EGARCH, Kalman EM, HMM, OLS du VRP, moyennes/écarts
# de normalisation) sont fittés sur le train du PREMIER fold uniquement (FIT_IDX) :
# comme tous les folds walk-forward ont un train qui commence au début de
# l'historique, aucune de ces estimations ne voit jamais une période de test.
print("[FEATURES] Démarrage (fit des estimateurs sur le train du 1er fold)...")
df_features = build_features(df_raw, VIX_COL, SPX_COL, FIT_IDX)
print(f"Dataset: {df_features.shape}")


## Nouvelles features ciblées « spike » (fortes hausses du VIX)

Le modèle GLOBAL h=5j validé en walk-forward est robuste sur la direction (F1_dir≈0.61) mais **rate les fortes hausses** (F1_UP_FORT≈0.36). Ces features visent spécifiquement les précurseurs de spike de volatilité. **Toutes sont causales par construction** (ratios auto-normalisés ou z-scores sur fenêtre glissante) : chaque valeur à la date *t* n'utilise que le passé, donc aucune fuite train/test — pas besoin de les fitter sur le train.

### Rough volatility — exposant de Hurst (Gatheral, Jaisson & Rosenbaum, 2018)
La volatilité est « rugueuse » : l'exposant de Hurst $H$ des log-variances est empiriquement $\approx 0.1 \ll 0.5$. Un $H$ faible = trajectoire très irrégulière, forte anti-persistance = régime propice aux sauts. Estimé en fenêtre glissante via la loi d'échelle de la variance agrégée : pour des blocs de taille $L$, $\operatorname{Var}(\sum_L r) \propto L^{2H}$, d'où $H = \tfrac12\,\text{pente}\big(\log \operatorname{Var}(\sum_L r)\ \text{vs}\ \log L\big)$.

### Semi-variance réalisée haussière (Barndorff-Nielsen et al., 2010)
Décomposition de la variance réalisée en parts haussière/baissière : $RS^+ = \sum_{t} r_t^2\,\mathbb{1}(r_t>0)$. Le ratio $RS^+/RV$ mesure l'**asymétrie de la variance vers le haut** — une variance dominée par les hausses du VIX précède les spikes.

### Structure par terme du VIX (contango / backwardation)
En régime calme, la courbe VIX est en *contango* (VIX9D < VIX < VIX3M). Un passage en *backwardation* (front > long, ratio > 1) est un signal de stress classique. Features : $\text{VIX9D}/\text{VIX}$ et $\text{VIX}/\text{VIX3M}$ (ratios, sans normalisation, donc causaux). Construites seulement si `^VIX9D` / `^VIX3M` sont disponibles.

### Indice SKEW du CBOE (proxy options-flow / risque de queue)
Le SKEW mesure le prix payé pour les options SPX très en dehors de la monnaie (protection contre les krachs) : SKEW élevé = le marché price une queue gauche épaisse = demande de couverture. Features : niveau, variation à 5j, z-score glissant 252j. Construites seulement si `^SKEW` disponible.

### Prime de vol-of-vol (VVIX/VIX) et sauts signés
- **VVIX/VIX** : la volatilité du VIX rapportée à son niveau ; une vol-of-vol élevée relativement au VIX annonce l'instabilité.
- **Intensité de sauts haussiers** : fraction de jours où $r_t > 2\hat\sigma_{60}$ sur 20j, et asymétrie sauts hauts − sauts bas — mesure directe de la fréquence récente des chocs à la hausse.


In [ ]:
# ============================================================
# FEATURES SPIKE (toutes causales — aucune statistique fittée sur le train)
# ============================================================
def _hurst_window(x):
    """Exposant de Hurst d'une fenêtre de log-rendements via la loi d'échelle
    de la variance agrégée : Var(somme de L rendements) ∝ L^(2H)."""
    x = x[~np.isnan(x)]
    if len(x) < 40:
        return np.nan
    lags = [1, 2, 4, 8]; v = []
    for L in lags:
        n = len(x) // L
        if n < 4:
            return np.nan
        agg = x[:n * L].reshape(n, L).sum(axis=1)
        v.append(np.var(agg))
    v = np.array(v)
    if np.any(v <= 0):
        return np.nan
    slope = np.polyfit(np.log(lags), np.log(v), 1)[0]
    return slope / 2.0

def add_spike_features(df, vix_col):
    """Ajoute les features spike à df (in place) et renvoie la liste des noms ajoutés."""
    t0 = time.time(); added = []
    vix = df[vix_col].replace([np.inf, -np.inf], np.nan).ffill().bfill()
    r = np.log(vix / vix.shift(1)).replace([np.inf, -np.inf], np.nan).fillna(0)

    def put(name, series):
        df[name] = series.replace([np.inf, -np.inf], np.nan)
        added.append(name)

    # 1. Semi-variance haussière (ratio RS+/RV), causal
    for w in [20, 60]:
        up = (r.clip(lower=0) ** 2).rolling(w, min_periods=w // 2).sum()
        tot = (r ** 2).rolling(w, min_periods=w // 2).sum().replace(0, np.nan)
        put(f'spike_semivar_up_ratio_{w}d', up / tot)

    # 2. Sauts signés (causal) : seuil 2×écart-type glissant 60j
    sig60 = r.rolling(60, min_periods=30).std()
    up_j = (r > 2 * sig60).astype(float); dn_j = (r < -2 * sig60).astype(float)
    put('spike_up_jump_int_20d', up_j.rolling(20, min_periods=10).mean())
    put('spike_jump_asym_20d', (up_j - dn_j).rolling(20, min_periods=10).mean())

    # 3. Rough volatility — Hurst glissant (causal)
    put('spike_hurst_120d', pd.Series(r.values, index=r.index)
        .rolling(120, min_periods=80).apply(_hurst_window, raw=True))

    # 4. Structure par terme du VIX (backwardation) — si disponibles
    v9 = [c for c in df.columns if 'VIX9D' in c]
    v3 = [c for c in df.columns if 'VIX3M' in c]
    if v9:
        s9 = df[v9[0]].replace([np.inf, -np.inf], np.nan).ffill()
        put('spike_ts_9d_over_vix', s9 / vix.replace(0, np.nan))
    if v3:
        s3 = df[v3[0]].replace([np.inf, -np.inf], np.nan).ffill()
        put('spike_vix_over_3m', vix / s3.replace(0, np.nan))

    # 5. Indice SKEW (options-flow / risque de queue) — si disponible
    sk = [c for c in df.columns if 'SKEW' in c]
    if sk:
        s = df[sk[0]].replace([np.inf, -np.inf], np.nan).ffill()
        put('spike_skew_level', s)
        put('spike_skew_ret_5d', s.pct_change(5))
        mu = s.rolling(252, min_periods=120).mean(); sd = s.rolling(252, min_periods=120).std().replace(0, np.nan)
        put('spike_skew_z_252d', (s - mu) / sd)

    # 6. Prime de vol-of-vol VVIX/VIX — si VVIX disponible
    vv = [c for c in df.columns if 'VVIX' in c]
    if vv:
        vvix = df[vv[0]].replace([np.inf, -np.inf], np.nan).ffill()
        ratio = vvix / vix.replace(0, np.nan)
        put('spike_vvix_vix_ratio', ratio)
        mu = ratio.rolling(252, min_periods=120).mean(); sd = ratio.rolling(252, min_periods=120).std().replace(0, np.nan)
        put('spike_vvix_vix_z_252d', (ratio - mu) / sd)

    print(f"  [SPIKE] {len(added)} features ajoutées: {added} ({time.time()-t0:.0f}s)")
    return added

SPIKE_FEATURES = add_spike_features(df_features, VIX_COL)
print(f"Dataset enrichi: {df_features.shape}")


In [ ]:
def build_target(vix_series, horizon, split_idx):
    """Cible 4 classes à seuils conditionnels au régime, quantiles fittés
    uniquement sur les split_idx premières observations (train du fold)."""
    vix = vix_series.ffill().bfill(); vix_tr = vix.iloc[:split_idx]
    calm_thr = vix_tr.quantile(0.33); stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr] = 'CALM'; regime[vix >= stress_thr] = 'STRESS'
    ret = (vix.shift(-horizon) / vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat].dropna(); reg_r = regime.reindex(ret.index)
    cut_date = vix.index[min(split_idx, len(vix) - 1)]
    ret_tr = ret.loc[ret.index < cut_date]; reg_tr = reg_r.loc[ret_tr.index]
    thr = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_tr[reg_tr == reg]
        thr[reg] = (sub.quantile(0.25) if len(sub) >= 20 else ret_tr.quantile(0.25),
                    sub.quantile(0.75) if len(sub) >= 20 else ret_tr.quantile(0.75))
    thr['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))
    def classify(r, reg):
        q25, q75 = thr.get(reg, (0, 0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3
    target = pd.Series([classify(r, reg_r[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    return target, reg_r, thr

def recon_feat(fname, df, w=20, eps=1e-8):
    """Reconstruit une feature d'interaction 'A__sep__B' à partir de ses deux
    composantes de base — mêmes 6 types que le rapport (§3.9, Table 4)."""
    if fname in df.columns: return df[fname]
    for sep in ['__div__', '__minus__', '__prod__', '__zrel__', '__macross__', '__ret5x__']:
        if sep in fname:
            a, b = fname.split(sep, 1)
            if a not in df.columns or b not in df.columns: return None
            si, sj = df[a], df[b]
            if sep == '__div__':    return si / sj.where(sj.abs() >= eps, np.nan)
            if sep == '__minus__':  return si - sj
            if sep == '__prod__':   return si * sj
            if sep == '__zrel__':
                d = si - sj; rs = d.rolling(w, min_periods=w // 2).std(); return d / rs.replace(0, np.nan)
            if sep == '__macross__':
                mi = si.rolling(w, min_periods=w // 2).mean(); mj = sj.rolling(w, min_periods=w // 2).mean()
                return mi / mj.where(mj.abs() >= eps, np.nan)
            if sep == '__ret5x__':  return si.pct_change(5) * sj
    return None

def build_X(df, feats):
    cols = {}
    for f in feats:
        s = recon_feat(f, df)
        if s is not None: cols[f] = s
    return pd.DataFrame(cols, index=df.index).replace([np.inf, -np.inf], np.nan) if cols else pd.DataFrame(index=df.index)

def shap_importance(X, y, prefilter=None):
    """|SHAP| moyen par colonne de X (numpy 2D), robuste à la disposition
    (n_obs,n_features,n_classes) ou (n_classes,n_obs,n_features)."""
    nf = X.shape[1]
    if prefilter and nf > prefilter:
        pf = XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                           eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
        pf.fit(X, y); keep = np.argsort(pf.feature_importances_)[::-1][:prefilter]
    else:
        keep = np.arange(nf)
    Xk = X[:, keep]
    pilot = XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                          eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
    pilot.fit(Xk, y)
    sv = np.abs(np.array(shap.TreeExplainer(pilot).shap_values(Xk[:min(CONFIG['shap_sample'], len(Xk))])))
    nfk = Xk.shape[1]
    feat_axes = [ax for ax in range(sv.ndim) if sv.shape[ax] == nfk]
    if len(feat_axes) == 1:
        arr = sv.mean(axis=tuple(ax for ax in range(sv.ndim) if ax != feat_axes[0]))
    else:
        arr = np.asarray(pilot.feature_importances_)
    full = np.zeros(nf); full[keep] = np.asarray(arr).ravel()
    return full

print("Helpers OK (build_target, recon_feat/build_X, shap_importance)")


## Interactions de features inter-tickers (méthodologie du rapport, §3.9)

**Principe.** Certaines relations *entre* deux séries (deux tickers, ou un ticker et le VIX) portent un signal que ni l'une ni l'autre ne porte seule — par exemple un ratio de vol relative entre secteurs, ou un écart de tendance entre deux actifs corrélés en temps normal. On génère systématiquement 6 transformations par paire de features candidates (ratio, différence, produit, z-score relatif, moyenne mobile croisée, momentum croisé — cf. `recon_feat`), puis on **sélectionne** celles qui portent réellement du signal via SHAP.

**Deux passes** (pour ne pas générer $O(n^2)$ interactions sur tout le pool, ce qui serait ingérable) :
1. **Passe 1** — un XGBoost pilote (SHAP) classe les features de *base* (engineered, hors interactions) ; on garde le top-40.
2. **Passe 2** — parmi ce top-40, seules les 20 plus importantes servent à générer les paires ($\binom{20}{2}=190$ paires × 6 transformations = 1140 interactions candidates). On les combine aux 40 features de base (1180 au total) et on refait un classement SHAP : les 30 meilleures sont conservées, et seules les interactions effectivement présentes dans ce top-30 sont ajoutées en tant que nouvelles colonnes permanentes du dataset.

**Choix méthodologique** : comme pour les paramètres du filtre de Kalman ou du HMM, cette sélection (quelles paires cross-multiplier) est une décision *structurelle* fixée une fois sur le train du premier fold walk-forward (`FIT_IDX`) — pas refaite à chaque fold. Cohérent avec le reste du pipeline : ça évite de régénérer/re-sélectionner 1140 colonnes à chaque fold (prohibitif), au prix d'une petite hypothèse de stabilité (les mêmes paires restent pertinentes tout au long de l'historique) — hypothèse déjà implicitement faite pour les autres estimateurs internes du pipeline.


In [ ]:
# ============================================================
# DÉCOUVERTE DES INTERACTIONS INTER-TICKERS (2 passes SHAP)
# ============================================================
BASE_POOL = [c for c in df_features.columns if c not in df_raw.columns]
print(f"Pool de base (avant interactions): {len(BASE_POOL)} features")

t0 = time.time()
tgt_pilot, reg_pilot, _ = build_target(df_features[VIX_COL], CONFIG['pilot_horizon'], FIT_IDX)
cut_date_pilot = all_dates[FIT_IDX]
tr_mask_pilot = np.asarray(tgt_pilot.index < cut_date_pilot)
y_pilot_tr = tgt_pilot.values[tr_mask_pilot].astype(int)

print("[INTERACTIONS] Passe 1 — SHAP sur les features de base...")
X_base_df = df_features[BASE_POOL].reindex(tgt_pilot.index)
X_base_tr = np.nan_to_num(X_base_df.values[tr_mask_pilot])
imp1 = shap_importance(X_base_tr, y_pilot_tr, prefilter=CONFIG['pool_prefilter'])
order1 = np.argsort(imp1)[::-1]
top_base = [BASE_POOL[i] for i in order1[:CONFIG['interact_top_base']]]
top_pairs_feats = top_base[:CONFIG['interact_top_pairs']]
print(f"  Top-{len(top_base)} identifiées ({time.time()-t0:.0f}s). "
      f"Top-{len(top_pairs_feats)} retenues pour les paires : {top_pairs_feats[:6]}...")

print("[INTERACTIONS] Passe 2 — génération des paires (6 types) + re-sélection...")
pairs = [(a, b) for i, a in enumerate(top_pairs_feats) for b in top_pairs_feats[i + 1:]]
SEPS = ['__div__', '__minus__', '__prod__', '__zrel__', '__macross__', '__ret5x__']
cand_names = [f"{a}{sep}{b}" for a, b in pairs for sep in SEPS]
print(f"  {len(pairs)} paires × {len(SEPS)} types = {len(cand_names)} interactions candidates")

X_cand = build_X(df_features, cand_names)  # matérialisé sur tout l'historique (opérations élémentaires, peu coûteux)
combined_names = top_base + list(X_cand.columns)
X_combined_df = pd.concat([df_features[top_base], X_cand], axis=1).reindex(tgt_pilot.index)
X_combined_tr = np.nan_to_num(X_combined_df.values[tr_mask_pilot])

imp2 = shap_importance(X_combined_tr, y_pilot_tr, prefilter=CONFIG['pool_prefilter'])
order2 = np.argsort(imp2)[::-1][:CONFIG['interact_final_n']]
top_final = [combined_names[i] for i in order2]
INTERACTION_FEATURES = [f for f in top_final if any(sep in f for sep in SEPS)]
print(f"  Top-{CONFIG['interact_final_n']} final : {len(INTERACTION_FEATURES)} interactions retenues "
      f"(le reste = features de base déjà présentes dans le pool)")

for f in INTERACTION_FEATURES:
    df_features[f] = X_cand[f]
print(f"[INTERACTIONS] {len(INTERACTION_FEATURES)} colonnes d'interaction ajoutées au dataset "
      f"({time.time()-t0:.0f}s) — dataset: {df_features.shape}")
if INTERACTION_FEATURES:
    print("  Exemples:", INTERACTION_FEATURES[:8])


In [ ]:
# ============================================================
# EXPORT DU DATASET ENRICHI (pour VIX_FINAL_ML_SCAN / VIX_FINAL_TFT / VIX_FINAL_OPTUNA)
# ============================================================
FEATURE_POOL = BASE_POOL + [c for c in df_features.columns if c not in BASE_POOL and c not in df_raw.columns]
meta = {
    'vix_col': VIX_COL, 'spx_col': SPX_COL,
    'feature_pool': FEATURE_POOL, 'interaction_features': INTERACTION_FEATURES,
    'config': CONFIG, 'notebook_version': NOTEBOOK_VERSION,
    'n_rows': len(df_features), 'n_cols': df_features.shape[1],
    'date_min': str(df_features.index.min().date()), 'date_max': str(df_features.index.max().date()),
}
df_features.to_parquet('vix_final_features.parquet')
with open('vix_final_features_meta.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)
print(f"[SAVE] vix_final_features.parquet ({df_features.shape}) + vix_final_features_meta.json")
print(f"  FEATURE_POOL: {len(FEATURE_POOL)} colonnes (dont {len(INTERACTION_FEATURES)} interactions)")

# ============================================================
# PUSH VERS GITHUB (nécessite le secret Colab GITHUB_TOKEN)
# Les notebooks ML_SCAN / TFT / OPTUNA le rechargent depuis cette branche —
# évite de refaire tout le data loading + feature engineering (le plus long)
# dans chacun d'eux.
# ============================================================
import subprocess

GITHUB_REPO = "LP-D/claude"
FEATURES_BRANCH = "results/vix-final-features"
RESULT_FILES = ['vix_final_features.parquet', 'vix_final_features_meta.json']

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

if not GITHUB_TOKEN:
    print("[SKIP] Secret 'GITHUB_TOKEN' introuvable — le dataset reste local à ce runtime Colab. "
          "Sans lui, VIX_FINAL_ML_SCAN/TFT/OPTUNA ne pourront pas le récupérer automatiquement "
          "(il faudra le transférer manuellement).")
else:
    workdir = "/content/_vix_features_push"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git", workdir],
                           capture_output=True, text=True)
    if clone.returncode != 0:
        print("[ERREUR] git clone a échoué :", clone.stderr[-1500:])
    else:
        exists = subprocess.run(["git", "-C", workdir, "ls-remote", "--exit-code", "--heads",
                                  "origin", FEATURES_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", workdir, "checkout", "-B", FEATURES_BRANCH,
                             f"origin/{FEATURES_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", workdir, "checkout", "-B", FEATURES_BRANCH], check=True)
        for fn in RESULT_FILES:
            subprocess.run(["cp", fn, f"{workdir}/{fn}"], check=True)
        subprocess.run(["git", "-C", workdir, "config", "user.email", "vix-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", workdir, "config", "user.name", "VIX Final Features Colab run"], check=True)
        subprocess.run(["git", "-C", workdir, "add"] + RESULT_FILES, check=True)
        commit = subprocess.run(["git", "-C", workdir, "commit", "-m",
                                 f"Dataset features — {pd.Timestamp.now():%Y-%m-%d %H:%M} ({meta['n_rows']}x{meta['n_cols']})"],
                                capture_output=True, text=True)
        print(commit.stdout or commit.stderr)
        push = subprocess.run(["git", "-C", workdir, "push", "origin", FEATURES_BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[PUSH OK] Dataset disponible sur la branche '{FEATURES_BRANCH}' de {GITHUB_REPO}")
        else:
            print("[ERREUR] git push a échoué :", push.stderr[-1500:])
